# Feature engineering to train models


In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import os

base_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
db_path = os.path.join(base_dir, "data", "electricity.db")
conn = sqlite3.connect(db_path)
# Filtramos cambio de hora esta vez
df = pd.read_sql("""
    SELECT * FROM omie_prices
    WHERE hour BETWEEN 1 AND 24  
    ORDER BY datetime
""", conn)

print(df.shape)

(8735, 7)


### Variables temporales cílicas lo primero

In [2]:
# Variables cíclicas de hora del día
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

# Variables cíclicas de día del año
df["datetime"] = pd.to_datetime(df["datetime"])
df["day_of_year"] = df["datetime"].dt.dayofyear
df["doy_sin"] = np.sin(2 * np.pi * df["day_of_year"] / 365)
df["doy_cos"] = np.cos(2 * np.pi * df["day_of_year"] / 365)

# Día de la semana (0=lunes en pandas, distinto a SQLite)
df["weekday"] = df["datetime"].dt.dayofweek
df["weekday_sin"] = np.sin(2 * np.pi * df["weekday"] / 7)
df["weekday_cos"] = np.cos(2 * np.pi * df["weekday"] / 7)

In [3]:
df.head()

,year,month,day,hour,price_es,price_pt,datetime,hour_sin,hour_cos,day_of_year,doy_sin,doy_cos,weekday,weekday_sin,weekday_cos
0,2023,1.0,1.0,1.0,0.0,0.0,2023-01-01 00:00:00,0.258819,0.965926,1,0.017213,0.999852,6,-0.781831,0.62349
1,2023,1.0,1.0,2.0,0.0,0.0,2023-01-01 01:00:00,0.500000,0.866025,1,0.017213,0.999852,6,-0.781831,0.62349
2,2023,1.0,1.0,3.0,0.0,0.0,2023-01-01 02:00:00,0.707107,0.707107,1,0.017213,0.999852,6,-0.781831,0.62349
3,2023,1.0,1.0,4.0,0.0,0.0,2023-01-01 03:00:00,0.866025,0.500000,1,0.017213,0.999852,6,-0.781831,0.62349
4,2023,1.0,1.0,5.0,0.0,0.0,2023-01-01 04:00:00,0.965926,0.258819,1,0.017213,0.999852,6,-0.781831,0.62349


### Ahora las Lag features (hora, dia, semana)
Para ello importantísimo primero ordenar los datos segun datetime

In [ ]:
df = df.sort_values("datetime").reset_index(drop=True)
# Reseteamos indices y quitamos indices antiguos (drop = True)

In [5]:
# Lag features

df["lag_1"] = df["price_es"].shift(1)      # precio hora anterior
df["lag_24"] = df["price_es"].shift(24)    # precio misma hora ayer
df["lag_168"] = df["price_es"].shift(168)  # precio misma hora hace una semana

# Eliminar filas con NaN generadas por los lags
df = df.dropna().reset_index(drop=True)

print(f"Filas tras eliminar NaN: {len(df)}")

Filas tras eliminar NaN: 8567


### A continuación vamos a las 'medias moviles' (Rolling data)
Las medias móviles (rolling means) calculan el precio medio de las últimas N horas en cada instante del tiempo. Por ejemplo, rolling_mean_24 para la hora 14:00 del 15 de enero es el precio medio de las 24 horas anteriores — del 14 de enero a las 15:00 hasta el 15 de enero a las 13:00. Esto captura la tendencia reciente del precio y es muy útil para el modelo porque si el precio ha sido alto las últimas 24 horas, probablemente siga alto. El shift(1) antes del rolling es fundamental — desplaza los valores una posición hacia atrás para garantizar que la media solo incluye horas pasadas y nunca la hora actual, evitando así data leakage (filtración de información futura al modelo).


In [6]:
# Medias móviles
df["rolling_mean_24"] = df["price_es"].shift(1).rolling(window=24).mean()   # media últimas 24h
df["rolling_mean_168"] = df["price_es"].shift(1).rolling(window=168).mean() # media última semana

# Eliminar NaN nuevos
df = df.dropna().reset_index(drop=True)

print(f"Filas tras medias móviles: {len(df)}")

Filas tras medias móviles: 8399


### Por ultimo variable binaria de fin de semana
En un futuro quizas extender a festivos nacionales con calendario

In [ ]:
df["is_weekend"] = (df["weekday"] >= 5).astype(int)
# Devuelve 1 si es finde y 0 si no lo es

In [10]:
# Guardar features en SQLite
conn2 = sqlite3.connect(db_path)
df.to_sql("omie_features", conn2, if_exists="replace", index=False)
print(f"Tabla omie_features creada con {len(df)} filas y {len(df.columns)} columnas")
conn2.close()

Tabla omie_features creada con 8399 filas y 21 columnas
